In [1]:
from pathlib import Path

from imagematerials.__main__ import export_summary_netcdf, simulate_stocks
from imagematerials.vehicles.preprocessing import (
    preprocessing,
)
from imagematerials.util import import_from_netcdf, export_to_netcdf
from imagematerials.model import GenericMainModel, GenericMaterials, GenericStocks, Maintenance
from imagematerials.factory import ModelFactory
import prism


In [2]:
# import pandas as pd
# import xarray as xr
# from imagematerials.util import dataset_to_array
# maintenance_fp = Path("../data/raw/vehicles/standard_data/maintenance_passenger_cars.csv")
# df = pd.read_csv(maintenance_fp, index_col=0)
# xr.DataArray(df["total_material_per_km"], dims=("Material",), coords={"Material": df["Material"]})
# print(df)
# dataset_to_array(df.to_xarray(), ["Material"], [])


In [8]:
base_dir = "../data/raw"
prep_fp = Path("prep_vema2.nc")

In [9]:
if not prep_fp.is_file():
    _, orig_prep_data = preprocessing(base_dir)
    export_to_netcdf(orig_prep_data, prep_fp)
prep_data = import_from_netcdf(prep_fp)
prep_data["weights"] = prep_data.pop("vehicle_weights")

C:\Users\5982758\OneDrive - Universiteit Utrecht\Documents\Programming\image-materials\imagematerials\vehicles\preprocessing.py:331: PerformanceWarning: indexing past lexsort depth may impact performance.
  vehicle_shares_typical[('Cars', fuel)] = \
C:\Users\5982758\OneDrive - Universiteit Utrecht\Documents\Programming\image-materials\imagematerials\vehicles\preprocessing.py:331: PerformanceWarning: indexing past lexsort depth may impact performance.
  vehicle_shares_typical[('Cars', fuel)] = \
C:\Users\5982758\OneDrive - Universiteit Utrecht\Documents\Programming\image-materials\imagematerials\vehicles\preprocessing.py:331: PerformanceWarning: indexing past lexsort depth may impact performance.
  vehicle_shares_typical[('Cars', fuel)] = \
C:\Users\5982758\OneDrive - Universiteit Utrecht\Documents\Programming\image-materials\imagematerials\vehicles\preprocessing.py:331: PerformanceWarning: indexing past lexsort depth may impact performance.
  vehicle_shares_typical[('Cars', fuel)] = \


In [10]:
# Define the complete timeline, including historic tail
# time_start = prep_data["stocks"].coords["Time"].min().values
time_start = 1960
complete_timeline = prism.Timeline(time_start, 2060, 1)
simulation_timeline = prism.Timeline(1970, 2060, 1)

In [11]:
# Define the coordinates of all dimensions.
Region = list(prep_data["stocks"].coords["Region"].values)
Time = [t for t in complete_timeline]
Cohort = Time
Type = list(prep_data["stocks"].coords["Type"].values)
material = list(prep_data["material_fractions"].coords["material"].values)

# Create
main_model_normal = GenericMainModel(
    complete_timeline, Region=Region, Time=Time, Cohort=Cohort, Type=Type, prep_data=prep_data,
    compute_materials=True, compute_battery_materials=False, compute_maintenance_materials=True, 
    material=material)

In [13]:
main_model_factory = ModelFactory(
    prep_data, complete_timeline
    ).add(GenericStocks
    ).add(GenericMaterials
    ).add(Maintenance
    ).finish()

In [14]:
main_model_factory.simulate(simulation_timeline)

In [12]:
main_model_normal.simulate(simulation_timeline)

TypeError: Maintenance.compute_values() missing 3 required positional arguments: 'inflow', 'stock_by_cohort', and 'outflow_by_cohort'

In [10]:
# stocks_model.simulate(prism.Timeline(1970, 2060, 1))

In [11]:
# main_model = ModelFactory(prep_data).add("stock").add("material").run()

In [12]:
# main_model.outflow_by_cohort[2020].max()

In [13]:
# model = simulate_stocks(prep_data)

In [14]:
# export_summary_netcdf(model, "data_sums.nc")